# RCPSP-AS: OptalCP Formulations

Two CP formulations for the Resource-Constrained Project Scheduling Problem with Alternative Subgraphs, implemented with [OptalCP](https://optalcp.com/).

- **Simplified** — selection propagation via non-branching arcs $A_{\text{prop}}$
- **Original** — explicit activity selection via branch membership map $B_i$

Both formulations use the instance parser and model builders from `solve_rcpspas.py`.

In [15]:
import optalcp as cp
from solve_rcpspas import load_instance, _build_branch_map

## Load Instance

In [16]:
instance = load_instance('../../data/rcpspas/ASLIB/ASLIB0/aslib0_0a.RCP')
act_dict = {act.id: act for act in instance.activities}
n = len(instance.activities)
print(f"{instance.name}: {n} activities, {len(instance.resources)} resources, "
      f"{len(instance.subgraphs)} subgraphs")

aslib0_0: 122 activities, 5 resources, 2 subgraphs


## Simplified Formulation

Uses the set of non-branching arcs $A_{\text{prop}} = A \setminus \{(p_l, j) \in A : l \in L\}$. Selection propagates along $A_{\text{prop}}$ from the always-present source, automatically handling fixed activities, within-branch successors, and cross-branch links.

In [17]:
# Identify branching arcs (excluded from A_prop)
branching_arcs = {
    (sub.principal_activity, s)
    for sub in instance.subgraphs if sub.principal_activity in act_dict
    for s in act_dict[sub.principal_activity].successors if s in act_dict
}

mdl = cp.Model()

# (30) Optional interval variable per activity
x = {i: mdl.interval_var(name=f"T_{i}", optional=True, length=act.duration)
     for i, act in act_dict.items()}

# (24) Minimize makespan
mdl.minimize(x[n - 1].end())

# (25) Source activity is present
mdl.enforce(x[0].presence() == 1)

# (26) Selection propagation along A_prop
for act in instance.activities:
    for j in act.successors:
        if j in x and (act.id, j) not in branching_arcs:
            mdl.enforce(x[act.id].presence().implies(x[j].presence()))

# (27) Branch selection: exactly one successor of p_l chosen iff p_l is present
for sub in instance.subgraphs:
    if sub.principal_activity in act_dict:
        succs = [s for s in act_dict[sub.principal_activity].successors if s in x]
        if succs:
            mdl.enforce(sum(x[s].presence() for s in succs) ==
                        x[sub.principal_activity].presence())

# (28) Precedence timing for all arcs
for act in instance.activities:
    for j in act.successors:
        if j in x:
            mdl.enforce((x[act.id].presence() & x[j].presence()).implies(
                x[act.id].end() <= x[j].start()))

# (29) Resource capacity constraints
for v, capacity in enumerate(instance.resources):
    if capacity > 0:
        pulses = [x[act.id].pulse(height=act.requirements[v])
                  for act in instance.activities if act.requirements[v] > 0]
        if pulses:
            mdl.enforce(mdl.sum(pulses) <= capacity)

In [18]:
result_simplified = mdl.solve(cp.Parameters(timeLimit=30))
print(f"Simplified: objective={result_simplified.objective}, optimal={result_simplified.proof}")

────────────────────────────────────────────────────────────────────────────────
                         ScheduleOpt OptalCP [Academic]
                            Version 2026.1.0 (Linux)
                CPU: AMD Ryzen 5 5500U with (12 physical cores)
────────────────────────────────────────────────────────────────────────────────
Input parse time: 00:00
   nbWorkers = 12                     (auto: 12 physical cores)
   preset = Default                   (auto: < 100,000 variables)
   timeLimit = 30 seconds
   noOverlapPropagationLevel = 4      (preset: Default)
   cumulPropagationLevel = 3          (preset: Default)
    Workers 0-5: searchType = LNS     (preset: Default)
    Workers 6-8: searchType = FDS     (preset: Default)
   Workers 9-11: searchType = FDSDual (preset: Default)
Input:
   0 integer variables, 122 interval variables, 438 constraints, 213kB
   00:00 Presolving..
Presolved:
   0 integer variables, 122 interval variables, 437 constraints, 274kB
   00:00 Presolve compl

## Original Formulation

Uses the branch membership map $M$: `branch_id` $\to$ `branching_activity_id` (with a dummy branch $k^*$ mapping to the source). Activity $i$ is present iff any branch it belongs to is selected:

$$\mathrm{presenceOf}(\mathsf{x}_i) \;\Leftrightarrow\; \bigvee_{k \in B_i} \mathrm{presenceOf}(\mathsf{x}_{b_k})$$

In [19]:
# Build M: branch_id -> branching_activity_id (branch 0 -> activity 0)
M = _build_branch_map(instance)

mdl2 = cp.Model()

# (16) Optional interval variable per activity
y = {i: mdl2.interval_var(name=f"T_{i}", optional=True, length=act.duration)
     for i, act in act_dict.items()}

# (10) Minimize makespan
mdl2.minimize(y[n - 1].end())

# (11) Source activity is present
mdl2.enforce(y[0].presence() == 1)

# (12) Activity selection via branch membership
for i, act in act_dict.items():
    if i != 0:
        branch_presences = [y[M[b_id]].presence()
                            for b_id in act.branches if b_id in M]
        if branch_presences:
            mdl2.enforce(y[i].presence() == (sum(branch_presences) > 0))

# (13) Subgraph branch selection
for sub in instance.subgraphs:
    if sub.principal_activity in act_dict:
        branches = [s for s in act_dict[sub.principal_activity].successors if s in y]
        if branches:
            mdl2.enforce(sum(y[s].presence() for s in branches) ==
                         y[sub.principal_activity].presence())

# (14) Precedence relations for all arcs
for act in instance.activities:
    for j in act.successors:
        if j in y:
            mdl2.enforce((y[act.id].presence() & y[j].presence()).implies(
                y[act.id].end() <= y[j].start()))

# (15) Resource capacity constraints
for v, capacity in enumerate(instance.resources):
    if capacity > 0:
        pulses = [y[act.id].pulse(height=act.requirements[v])
                  for act in instance.activities if act.requirements[v] > 0]
        if pulses:
            mdl2.enforce(mdl2.sum(pulses) <= capacity)

In [20]:
result_original = mdl2.solve(cp.Parameters(timeLimit=30))
print(f"Original: objective={result_original.objective}, optimal={result_original.proof}")

────────────────────────────────────────────────────────────────────────────────
                         ScheduleOpt OptalCP [Academic]
                            Version 2026.1.0 (Linux)
                CPU: AMD Ryzen 5 5500U with (12 physical cores)
────────────────────────────────────────────────────────────────────────────────
Input parse time: 00:00
   nbWorkers = 12                     (auto: 12 physical cores)
   preset = Default                   (auto: < 100,000 variables)
   timeLimit = 30 seconds
   noOverlapPropagationLevel = 4      (preset: Default)
   cumulPropagationLevel = 3          (preset: Default)
    Workers 0-5: searchType = LNS     (preset: Default)
    Workers 6-8: searchType = FDS     (preset: Default)
   Workers 9-11: searchType = FDSDual (preset: Default)
Input:
   0 integer variables, 122 interval variables, 347 constraints, 225kB
   00:00 Presolving..
Presolved:
   0 integer variables, 122 interval variables, 309 constraints, 243kB
   00:00 Presolve compl

## Compare Results

In [21]:
print(f"Simplified: Cmax = {result_simplified.objective}")
print(f"Original:   Cmax = {result_original.objective}")

Simplified: Cmax = 100.0
Original:   Cmax = 100.0
